In [16]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

# =========================
# 1. Load Data
# =========================
print("Loading data...")
try:
    data = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Error: 'housing.csv' not found. Please make sure the dataset is in the same directory.")
    exit()

# Drop rows with missing values
data.dropna(inplace=True)
print("Data loaded and cleaned.")

# =========================
# 2. Feature Engineering
# =========================
print("Performing feature engineering for the main model...")
# Log transform skewed numerical features
data['total_rooms'] = np.log(data['total_rooms'] + 1)
data['total_bedrooms'] = np.log(data['total_bedrooms'] + 1)
data['population'] = np.log(data['population'] + 1)
data['households'] = np.log(data['households'] + 1)

# Create ratio features
data['bedroom_ratio'] = data['total_bedrooms'] / data['total_rooms']
data['household_rooms'] = data['total_rooms'] / data['households']

# =========================
# 3. Data Splitting
# =========================
X = data.drop(['median_house_value', 'ocean_proximity'], axis=1)
y = data['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================
# 4. Scaling
# =========================
print("Scaling features...")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

joblib.dump(scaler, 'housing_scaler.joblib')
print("Scaler saved as 'housing_scaler.joblib'")

# =========================
# 5. Hyperparameter Tuning with GridSearchCV
# =========================
print("\nStarting hyperparameter tuning with GridSearchCV...")
print("This may take several minutes...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 10],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

xgb_model = XGBRegressor(random_state=42)

# Set up GridSearchCV
# cv=3 means 3-fold cross-validation. n_jobs=-1 uses all available CPU cores.
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='r2')

# Fit the grid search to the data
grid_search.fit(X_train_s, y_train)

# Get the best model from the grid search
best_xgb_model = grid_search.best_estimator_

print("\nBest Hyperparameters Found:")
print(grid_search.best_params_)

# =========================
# 6. Evaluation with the Best Model
# =========================
print("\nEvaluating the best model on the test set...")
y_pred_best = best_xgb_model.predict(X_test_s)
print("\nBest XGBoost Model Evaluation:")
print(f"R² Score: {r2_score(y_test, y_pred_best):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}")

# =========================
# 7. Save the Best Model
# =========================
joblib.dump(best_xgb_model, 'housing_xgb_model.joblib')
print("\nBest XGBoost model saved as 'housing_xgb_model.joblib'")

print("\nImproved model training complete!")


Loading data...
Data loaded and cleaned.
Performing feature engineering for the main model...
Scaling features...
Scaler saved as 'housing_scaler.joblib'

Starting hyperparameter tuning with GridSearchCV...
This may take several minutes...
Fitting 3 folds for each of 72 candidates, totalling 216 fits

Best Hyperparameters Found:
{'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 300, 'subsample': 0.7}

Evaluating the best model on the test set...

Best XGBoost Model Evaluation:
R² Score: 0.8401
RMSE: 46767.42

Best XGBoost model saved as 'housing_xgb_model.joblib'

Improved model training complete!
